In [0]:
import sys
import os

# Get the current notebook path dynamically (Databricks workspace path)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
config_dir = '/Workspace' + os.path.dirname(os.path.dirname(notebook_path))
sys.path.append(config_dir)

import datetime
import json
import utils

# create a requests session with retries/backoff
session = utils.create_session()


In [0]:
# Read all saved location files (wildcard) to avoid relying on a single filename
df = spark.read.json('/Volumes/openaq/bronze_dev/locations/*')
result_list = df.select("sensors", "id").toPandas().to_dict(orient='records')

In [0]:
import datetime
from pathlib import Path

# fetch hourly measurements for the last 24 hours by default
dt_to = datetime.datetime.utcnow()
dt_from = dt_to - datetime.timedelta(days=1)
datetime_to = dt_to.replace(microsecond=0).isoformat() + 'Z'
datetime_from = dt_from.replace(microsecond=0).isoformat() + 'Z'

for item in result_list:
    location_id = item.get('id')
    sensors = item.get('sensors', [])
    for sensor in sensors:
        sensor_id = sensor.get('id')
        if not sensor_id:
            continue
        measurements = utils.get_sensor_measurements(session, sensor_id, datetime_from=datetime_from, datetime_to=datetime_to, limit=100)
        timestamp = datetime.datetime.utcnow().strftime("%Y-%m-%d_%H-%M-%S")
        dir_path = Path(f'/Volumes/openaq/bronze_dev/sensor_data/{location_id}_{sensor_id}')
        dir_path.mkdir(parents=True, exist_ok=True)
        with open(dir_path / f'{sensor_id}_{timestamp}.json', 'w') as f:
            f.write(json.dumps(measurements))

In [0]:
from pathlib import Path

# Check the size of every nested file in /Volumes/openaq/bronze_dev/sensor_data
sensor_data_dir = Path('/Volumes/openaq/bronze_dev/sensor_data')
file_sizes = []
for file_path in sensor_data_dir.glob('**/*.json'):
    file_sizes.append({'file': str(file_path), 'size_bytes': file_path.stat().st_size})

display(file_sizes)